<a href="https://colab.research.google.com/github/ssk-algoverse/sae-binding/blob/main/circuit/VerySimpleInputPertubation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install transformer_lens

In [2]:
import torch

In [3]:
from huggingface_hub import hf_hub_download

REPO_ID = "sebastianhoenig/2L_1H_Entity_Binding"
FILENAME = "2L_1H_Attn_Only.pth"

weights_path = hf_hub_download(repo_id=REPO_ID, filename=FILENAME)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [39]:
### Model

from transformer_lens import HookedTransformer, HookedTransformerConfig

E = 100 # num entities
A = 100 # num attributes
T = 10 # num types/relations
SEP = E+A+T # as seperator between relations
Q = E+A+T+1 # question token
PAD = E+A+T+2
D_VOCAB = E+A+T+3
IGNORE_INDEX = -100

cfg = HookedTransformerConfig(
    n_layers=2,
    n_heads=1,
    d_model=256,
    d_head=256,
    d_mlp=1024,
    n_ctx=64,
    d_vocab=D_VOCAB,
    act_fn="gelu",
    attn_only=True,
    normalization_type="LN",
)
model = HookedTransformer(cfg)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Load the model
# Create a new model instance with the same configuration
pretrained_weights = torch.load(weights_path, map_location=device, weights_only=True)
model.load_state_dict(pretrained_weights)

print("Model loaded successfully.")

Moving model to device:  cuda
Model loaded successfully.


In [40]:
import numpy as np
import pandas as pd
import torch
import ast
from torch.utils.data import Dataset, DataLoader


In [41]:
id_mapping_df = pd.read_csv('id_mapping.csv')
id_to_entity = dict(zip(id_mapping_df['id'], id_mapping_df['name']))

In [42]:
class EntityBindingDataset(Dataset):
    def __init__(self, dataframe, parse_tokens_if_str=True):
        self.df = dataframe.reset_index(drop=True)
        self.parse_tokens_if_str = parse_tokens_if_str

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        seq = row["tokens"]
        tokens = torch.tensor(seq, dtype=torch.long)
        label  = torch.tensor(int(row["label"]), dtype=torch.long)
        return tokens, label

test_df = pd.read_csv('test_df.csv', converters={"tokens": ast.literal_eval})
test_dataset = EntityBindingDataset(test_df)

In [43]:
### Some very basic checks: What happens if we pertube the input sequence

In [44]:
example, label = test_dataset[0]

In [45]:
print("Example tokens and their mapped entities:")
for token_id in example:
    entity_name = id_to_entity.get(token_id.item(), f"Unknown Token {token_id.item()}")
    print(f"Token {token_id.item()} ({entity_name})")

print(f"\nLabel: {label.item()} ({id_to_entity.get(label.item(), f'Unknown Token {label.item()}')})")

Example tokens and their mapped entities:
Token 94 (Bryan)
Token 201 (was born in)
Token 118 (Istanbul)
Token 210 (Unknown Token 210)
Token 43 (Keith)
Token 208 (moved to)
Token 105 (Manila)
Token 210 (Unknown Token 210)
Token 18 (Lisa)
Token 200 (lives in)
Token 155 (Philadelphia)
Token 210 (Unknown Token 210)
Token 95 (Patrick)
Token 204 (studied in)
Token 144 (Riyadh)
Token 210 (Unknown Token 210)
Token 48 (Cynthia)
Token 202 (works in)
Token 106 (Shanghai)
Token 210 (Unknown Token 210)
Token 37 (Christine)
Token 205 (married in)
Token 111 (Sao Paulo)
Token 210 (Unknown Token 210)
Token 54 (Deborah)
Token 206 (visited)
Token 169 (Cape Town)
Token 210 (Unknown Token 210)
Token 80 (Anna)
Token 207 (loves)
Token 154 (Miami)
Token 210 (Unknown Token 210)
Token 208 (moved to)
Token 43 (Keith)
Token 211 (Unknown Token 211)

Label: 105 (Manila)


In [46]:
with torch.no_grad():
  logits = model(example)

In [47]:
pred = logits[0, -1, :].argmax().item()
print(f"Pred: {pred} ({id_to_entity.get(pred, f'Unknown Token {pred}')})")

Pred: 105 (Manila)


In [48]:
all_logits = logits[0, -1, :]
all_probs = torch.softmax(all_logits, dim=0)

# Get the top 10 logit values and their indices
top_logits, top_indices = torch.topk(all_logits, 10)

print("Top 10 predictions, logits, and probabilities (softmax over all logits):")
# Iterate through the top indices to get corresponding probabilities
for i in range(len(top_indices)):
    idx = top_indices[i]
    logit = top_logits[i]
    prob = all_probs[idx] # Get probability from the full softmax tensor
    entity_name = id_to_entity.get(idx.item(), f"Unknown Token {idx.item()}")
    print(f"Token {idx.item()} ({entity_name}) → Logit: {logit.item():.4f}, Probability: {prob.item():.2%}")

Top 10 predictions, logits, and probabilities (softmax over all logits):
Token 105 (Manila) → Logit: 11.5813, Probability: 63.14%
Token 106 (Shanghai) → Logit: 10.4718, Probability: 20.82%
Token 155 (Philadelphia) → Logit: 9.3648, Probability: 6.88%
Token 169 (Cape Town) → Logit: 8.9561, Probability: 4.57%
Token 118 (Istanbul) → Logit: 8.9376, Probability: 4.49%
Token 111 (Sao Paulo) → Logit: 3.7446, Probability: 0.02%
Token 154 (Miami) → Logit: 2.7084, Probability: 0.01%
Token 151 (Houston) → Logit: 2.2000, Probability: 0.01%
Token 129 (Bangkok) → Logit: 1.9961, Probability: 0.00%
Token 113 (Karachi) → Logit: 1.9321, Probability: 0.00%


In [49]:
### Changing the 105 token

In [50]:
example_2 = example.clone()
example_2[6] = 123
example_2
print("Example with Token 6 perturbation (Manila to Lahore):")
for token_id in example_2:
    entity_name = id_to_entity.get(token_id.item(), f"Unknown Token {token_id.item()}")
    print(f"Token {token_id.item()} ({entity_name})")

print(f"\nLabel: {label.item()} ({id_to_entity.get(label.item(), f'Unknown Token {label.item()}')})")

Example with Token 6 perturbation (Manila to Lahore):
Token 94 (Bryan)
Token 201 (was born in)
Token 118 (Istanbul)
Token 210 (Unknown Token 210)
Token 43 (Keith)
Token 208 (moved to)
Token 123 (Lahore)
Token 210 (Unknown Token 210)
Token 18 (Lisa)
Token 200 (lives in)
Token 155 (Philadelphia)
Token 210 (Unknown Token 210)
Token 95 (Patrick)
Token 204 (studied in)
Token 144 (Riyadh)
Token 210 (Unknown Token 210)
Token 48 (Cynthia)
Token 202 (works in)
Token 106 (Shanghai)
Token 210 (Unknown Token 210)
Token 37 (Christine)
Token 205 (married in)
Token 111 (Sao Paulo)
Token 210 (Unknown Token 210)
Token 54 (Deborah)
Token 206 (visited)
Token 169 (Cape Town)
Token 210 (Unknown Token 210)
Token 80 (Anna)
Token 207 (loves)
Token 154 (Miami)
Token 210 (Unknown Token 210)
Token 208 (moved to)
Token 43 (Keith)
Token 211 (Unknown Token 211)

Label: 105 (Manila)


In [51]:
with torch.no_grad():
  logits_2 = model(example_2)
pred_2 = logits_2[0, -1, :].argmax().item()
print(f"Pred: {pred_2} ({id_to_entity.get(pred_2, f'Unknown Token {pred_2}')})")

Pred: 123 (Lahore)


In [52]:
all_logits = logits_2[0, -1, :]
all_probs = torch.softmax(all_logits, dim=0)

# Get the top 10 logit values and their indices
top_logits, top_indices = torch.topk(all_logits, 10)

print("Top 10 predictions, logits, and probabilities (softmax over all logits):")
# Iterate through the top indices to get corresponding probabilities
for i in range(len(top_indices)):
    idx = top_indices[i]
    logit = top_logits[i]
    prob = all_probs[idx] # Get probability from the full softmax tensor
    entity_name = id_to_entity.get(idx.item(), f"Unknown Token {idx.item()}")
    print(f"Token {idx.item()} ({entity_name}) → Logit: {logit.item():.4f}, Probability: {prob.item():.2%}")

Top 10 predictions, logits, and probabilities (softmax over all logits):
Token 123 (Lahore) → Logit: 13.5487, Probability: 92.09%
Token 155 (Philadelphia) → Logit: 10.3826, Probability: 3.88%
Token 106 (Shanghai) → Logit: 9.8046, Probability: 2.18%
Token 118 (Istanbul) → Logit: 9.3684, Probability: 1.41%
Token 169 (Cape Town) → Logit: 8.1298, Probability: 0.41%
Token 154 (Miami) → Logit: 4.0813, Probability: 0.01%
Token 111 (Sao Paulo) → Logit: 3.2971, Probability: 0.00%
Token 198 (Incheon) → Logit: 3.0494, Probability: 0.00%
Token 137 (Berlin) → Logit: 2.1183, Probability: 0.00%
Token 136 (Wuhan) → Logit: 2.0805, Probability: 0.00%


In [53]:
### Changing another random token
example_3 = example.clone()
example_3[10] = 123
example_3

print("Example with Token 10 perturbation (Philadephia to Lahore):")
for token_id in example_3:
    entity_name = id_to_entity.get(token_id.item(), f"Unknown Token {token_id.item()}")
    print(f"Token {token_id.item()} ({entity_name})")

with torch.no_grad():
  logits_3 = model(example_3)
pred_3 = logits_3[0, -1, :].argmax().item()
print(f"Pred: {pred_3} ({id_to_entity.get(pred_3, f'Unknown Token {pred_3}')})")

Example with Token 10 perturbation (Philadephia to Lahore):
Token 94 (Bryan)
Token 201 (was born in)
Token 118 (Istanbul)
Token 210 (Unknown Token 210)
Token 43 (Keith)
Token 208 (moved to)
Token 105 (Manila)
Token 210 (Unknown Token 210)
Token 18 (Lisa)
Token 200 (lives in)
Token 123 (Lahore)
Token 210 (Unknown Token 210)
Token 95 (Patrick)
Token 204 (studied in)
Token 144 (Riyadh)
Token 210 (Unknown Token 210)
Token 48 (Cynthia)
Token 202 (works in)
Token 106 (Shanghai)
Token 210 (Unknown Token 210)
Token 37 (Christine)
Token 205 (married in)
Token 111 (Sao Paulo)
Token 210 (Unknown Token 210)
Token 54 (Deborah)
Token 206 (visited)
Token 169 (Cape Town)
Token 210 (Unknown Token 210)
Token 80 (Anna)
Token 207 (loves)
Token 154 (Miami)
Token 210 (Unknown Token 210)
Token 208 (moved to)
Token 43 (Keith)
Token 211 (Unknown Token 211)
Pred: 105 (Manila)


In [54]:
top_values, top_indices = torch.topk(logits_3[0, -1, :], 3)
top_probs = torch.softmax(top_values, dim=0)
for idx, prob in zip(top_indices, top_probs):
    entity_name = id_to_entity.get(idx.item(), f"Unknown Token {idx.item()}")
    print(f"Token {idx.item()} ({entity_name}) → {prob.item():.2%}")

Token 105 (Manila) → 79.56%
Token 118 (Istanbul) → 15.87%
Token 106 (Shanghai) → 4.57%


In [55]:
### What happens if we change the entity or attribute - breaking the match

In [56]:
example_4 = example.clone()
example_4[4] = 48 # was 43 before

print("Example with Token 4 perturbation (Keith to Cynthia):")
for token_id in example_4:
    entity_name = id_to_entity.get(token_id.item(), f"Unknown Token {token_id.item()}")
    print(f"Token {token_id.item()} ({entity_name})")

with torch.no_grad():
  logits_4 = model(example_4)
pred_4 = logits_4[0, -1, :].argmax().item()
print(f"Pred: {pred_4} ({id_to_entity.get(pred_4, f'Unknown Token {pred_4}')})")

Example with Token 4 perturbation (Keith to Cynthia):
Token 94 (Bryan)
Token 201 (was born in)
Token 118 (Istanbul)
Token 210 (Unknown Token 210)
Token 48 (Cynthia)
Token 208 (moved to)
Token 105 (Manila)
Token 210 (Unknown Token 210)
Token 18 (Lisa)
Token 200 (lives in)
Token 155 (Philadelphia)
Token 210 (Unknown Token 210)
Token 95 (Patrick)
Token 204 (studied in)
Token 144 (Riyadh)
Token 210 (Unknown Token 210)
Token 48 (Cynthia)
Token 202 (works in)
Token 106 (Shanghai)
Token 210 (Unknown Token 210)
Token 37 (Christine)
Token 205 (married in)
Token 111 (Sao Paulo)
Token 210 (Unknown Token 210)
Token 54 (Deborah)
Token 206 (visited)
Token 169 (Cape Town)
Token 210 (Unknown Token 210)
Token 80 (Anna)
Token 207 (loves)
Token 154 (Miami)
Token 210 (Unknown Token 210)
Token 208 (moved to)
Token 43 (Keith)
Token 211 (Unknown Token 211)
Pred: 106 (Shanghai)


In [57]:
all_logits = logits_4[0, -1, :]
all_probs = torch.softmax(all_logits, dim=0)

# Get the top 10 logit values and their indices
top_logits, top_indices = torch.topk(all_logits, 10)

print("Top 10 predictions, logits, and probabilities (softmax over all logits):")
# Iterate through the top indices to get corresponding probabilities
for i in range(len(top_indices)):
    idx = top_indices[i]
    logit = top_logits[i]
    prob = all_probs[idx] # Get probability from the full softmax tensor
    entity_name = id_to_entity.get(idx.item(), f"Unknown Token {idx.item()}")
    print(f"Token {idx.item()} ({entity_name}) → Logit: {logit.item():.4f}, Probability: {prob.item():.2%}")

Top 10 predictions, logits, and probabilities (softmax over all logits):
Token 106 (Shanghai) → Logit: 12.5538, Probability: 53.97%
Token 105 (Manila) → Logit: 12.2720, Probability: 40.71%
Token 169 (Cape Town) → Logit: 10.1365, Probability: 4.81%
Token 155 (Philadelphia) → Logit: 7.6193, Probability: 0.39%
Token 118 (Istanbul) → Logit: 6.2121, Probability: 0.10%
Token 151 (Houston) → Logit: 2.3497, Probability: 0.00%
Token 100 (Tokyo) → Logit: 2.1882, Probability: 0.00%
Token 111 (Sao Paulo) → Logit: 2.1623, Probability: 0.00%
Token 198 (Incheon) → Logit: 2.0831, Probability: 0.00%
Token 154 (Miami) → Logit: 1.8944, Probability: 0.00%


In [58]:
# 48 (Cynthia) is used in another fact - and it seems that it now predicts both of these attributes - but 106 (Shanghai) with a larger likelihood
# even though the type relation is wrong -> maybe it has a bias towards examples used later in the sequence?

# Lets try editing again - with an entity thats not in the sequence

In [59]:
example_5 = example.clone()
example_5[4] = 5 # was 43 before

print("Example with Token 4 perturbation (Keith to Erica):")
for token_id in example_5:
    entity_name = id_to_entity.get(token_id.item(), f"Unknown Token {token_id.item()}")
    print(f"Token {token_id.item()} ({entity_name})")

with torch.no_grad():
  logits_5 = model(example_5)
pred_5 = logits_5[0, -1, :].argmax().item()
print(f"Pred: {pred_5} ({id_to_entity.get(pred_5, f'Unknown Token {pred_5}')})")

Example with Token 4 perturbation (Keith to Erica):
Token 94 (Bryan)
Token 201 (was born in)
Token 118 (Istanbul)
Token 210 (Unknown Token 210)
Token 5 (Erica)
Token 208 (moved to)
Token 105 (Manila)
Token 210 (Unknown Token 210)
Token 18 (Lisa)
Token 200 (lives in)
Token 155 (Philadelphia)
Token 210 (Unknown Token 210)
Token 95 (Patrick)
Token 204 (studied in)
Token 144 (Riyadh)
Token 210 (Unknown Token 210)
Token 48 (Cynthia)
Token 202 (works in)
Token 106 (Shanghai)
Token 210 (Unknown Token 210)
Token 37 (Christine)
Token 205 (married in)
Token 111 (Sao Paulo)
Token 210 (Unknown Token 210)
Token 54 (Deborah)
Token 206 (visited)
Token 169 (Cape Town)
Token 210 (Unknown Token 210)
Token 80 (Anna)
Token 207 (loves)
Token 154 (Miami)
Token 210 (Unknown Token 210)
Token 208 (moved to)
Token 43 (Keith)
Token 211 (Unknown Token 211)
Pred: 106 (Shanghai)


In [60]:
all_logits = logits_5[0, -1, :]
all_probs = torch.softmax(all_logits, dim=0)

# Get the top 10 logit values and their indices
top_logits, top_indices = torch.topk(all_logits, 10)

print("Top 10 predictions, logits, and probabilities (softmax over all logits):")
# Iterate through the top indices to get corresponding probabilities
for i in range(len(top_indices)):
    idx = top_indices[i]
    logit = top_logits[i]
    prob = all_probs[idx] # Get probability from the full softmax tensor
    entity_name = id_to_entity.get(idx.item(), f"Unknown Token {idx.item()}")
    print(f"Token {idx.item()} ({entity_name}) → Logit: {logit.item():.4f}, Probability: {prob.item():.2%}")

Top 10 predictions, logits, and probabilities (softmax over all logits):
Token 106 (Shanghai) → Logit: 12.4918, Probability: 54.01%
Token 105 (Manila) → Logit: 12.2200, Probability: 41.16%
Token 169 (Cape Town) → Logit: 9.9334, Probability: 4.18%
Token 155 (Philadelphia) → Logit: 7.7642, Probability: 0.48%
Token 118 (Istanbul) → Logit: 6.5362, Probability: 0.14%
Token 151 (Houston) → Logit: 2.3992, Probability: 0.00%
Token 111 (Sao Paulo) → Logit: 2.3263, Probability: 0.00%
Token 100 (Tokyo) → Logit: 2.1390, Probability: 0.00%
Token 198 (Incheon) → Logit: 2.0659, Probability: 0.00%
Token 154 (Miami) → Logit: 2.0567, Probability: 0.00%


In [61]:
## almost the same???  -> discuss
# Now if we change the question entity as well, it should change again

In [62]:
example_6 = example.clone()
example_6[4] = 5 # was 43 (Keith) before
example_6[-2] = 5 # was 43 (Keith) before

print("Example with Token 4 and 33 perturbation (Keith to Erica):")
for token_id in example_6:
    entity_name = id_to_entity.get(token_id.item(), f"Unknown Token {token_id.item()}")
    print(f"Token {token_id.item()} ({entity_name})")

with torch.no_grad():
  logits_6 = model(example_6)
pred_6 = logits_6[0, -1, :].argmax().item()
print(f"Pred: {pred_6} ({id_to_entity.get(pred_6, f'Unknown Token {pred_6}')})")

Example with Token 4 and 33 perturbation (Keith to Erica):
Token 94 (Bryan)
Token 201 (was born in)
Token 118 (Istanbul)
Token 210 (Unknown Token 210)
Token 5 (Erica)
Token 208 (moved to)
Token 105 (Manila)
Token 210 (Unknown Token 210)
Token 18 (Lisa)
Token 200 (lives in)
Token 155 (Philadelphia)
Token 210 (Unknown Token 210)
Token 95 (Patrick)
Token 204 (studied in)
Token 144 (Riyadh)
Token 210 (Unknown Token 210)
Token 48 (Cynthia)
Token 202 (works in)
Token 106 (Shanghai)
Token 210 (Unknown Token 210)
Token 37 (Christine)
Token 205 (married in)
Token 111 (Sao Paulo)
Token 210 (Unknown Token 210)
Token 54 (Deborah)
Token 206 (visited)
Token 169 (Cape Town)
Token 210 (Unknown Token 210)
Token 80 (Anna)
Token 207 (loves)
Token 154 (Miami)
Token 210 (Unknown Token 210)
Token 208 (moved to)
Token 5 (Erica)
Token 211 (Unknown Token 211)
Pred: 105 (Manila)


In [63]:
all_logits = logits_6[0, -1, :]
all_probs = torch.softmax(all_logits, dim=0)

# Get the top 10 logit values and their indices
top_logits, top_indices = torch.topk(all_logits, 10)

print("Top 10 predictions, logits, and probabilities (softmax over all logits):")
# Iterate through the top indices to get corresponding probabilities
for i in range(len(top_indices)):
    idx = top_indices[i]
    logit = top_logits[i]
    prob = all_probs[idx] # Get probability from the full softmax tensor
    entity_name = id_to_entity.get(idx.item(), f"Unknown Token {idx.item()}")
    print(f"Token {idx.item()} ({entity_name}) → Logit: {logit.item():.4f}, Probability: {prob.item():.2%}")

Top 10 predictions, logits, and probabilities (softmax over all logits):
Token 105 (Manila) → Logit: 12.0737, Probability: 76.24%
Token 118 (Istanbul) → Logit: 10.0716, Probability: 10.30%
Token 106 (Shanghai) → Logit: 9.6761, Probability: 6.93%
Token 155 (Philadelphia) → Logit: 9.3132, Probability: 4.82%
Token 169 (Cape Town) → Logit: 8.2419, Probability: 1.65%
Token 111 (Sao Paulo) → Logit: 3.4405, Probability: 0.01%
Token 129 (Bangkok) → Logit: 2.5360, Probability: 0.01%
Token 154 (Miami) → Logit: 2.4145, Probability: 0.00%
Token 151 (Houston) → Logit: 2.1031, Probability: 0.00%
Token 198 (Incheon) → Logit: 1.9209, Probability: 0.00%
